In [2]:
import os
import time
import hashlib
import pandas as pd
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

# 1. Extraccion de datos

## 1.1 Web Scrappping X

In [ ]:
# Leer el token desde el archivo externo
ruta_token = "auth.tk"
try:
    with open(ruta_token, "r", encoding="utf-8") as archivo:
        mi_auth_token = archivo.read().strip()
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo '{ruta_token}'.")
    print("Crea el archivo en la misma carpeta del script, pega tu token dentro y vuelve a ejecutar.")
    driver.quit()
    exit()

In [ ]:
def seudonimizar_usuario(username):
    """Aplica un hash SHA-256 al nombre de usuario para cumplir con la Ley 21.719."""
    if not username: return None
    return hashlib.sha256(username.encode('utf-8')).hexdigest()

def extraer_comentarios_x(url, periodo, driver, limite=300):
    driver.get(url)
    print(f"Cargando {periodo}...")
    time.sleep(6) # Pausa inicial para carga de DOM
    
    comentarios_extraidos = []
    ids_vistos = set()
    intentos_sin_nuevos = 0 
    botones_cliqueados = set() # <--- Memoria para no repetir clics en el mismo botón
    
    while len(comentarios_extraidos) < limite:
        
        # Salvaguarda de navegación
        url_base = url.split('?')[0]
        if not driver.current_url.startswith(url_base):
            print("El navegador se desvió de la página principal. Regresando...")
            driver.get(url)
            time.sleep(5)
            
        cantidad_anterior = len(comentarios_extraidos)
        articulos = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')
        
        for articulo in articulos:
            try:
                # 1. Obtener enlace único del tweet
                link_elem = articulo.find_elements(By.XPATH, './/a[contains(@href, "/status/")]')
                tweet_id = link_elem[0].get_attribute('href') if link_elem else None
                
                if not tweet_id or tweet_id in ids_vistos:
                    continue
                
                # 2. Extraer texto del tweet
                texto_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetText"]')
                texto = texto_elem[0].text if texto_elem else None
                
                # 3. Detectar si contiene multimedia
                media_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetPhoto"] | .//div[@data-testid="videoPlayer"]')
                tiene_media = len(media_elem) > 0
                
                if not texto:
                    if tiene_media:
                        texto = "[Contenido multimedia sin texto]"
                    else:
                        continue 
                
                # 4. Extraer usuario y fecha
                user_elem = articulo.find_elements(By.XPATH, './/div[@dir="ltr"]/span')
                usuario_raw = user_elem[0].text if user_elem else "usuario_desconocido"
                
                time_elem = articulo.find_elements(By.XPATH, './/time')
                fecha_hora = time_elem[0].get_attribute('datetime') if time_elem else None
                
                ids_vistos.add(tweet_id)
                
                comentarios_extraidos.append({
                    "id_texto": len(ids_vistos),
                    "fecha_hora": fecha_hora,
                    "id_usuario_hash": seudonimizar_usuario(usuario_raw),
                    "texto": texto,
                    "consulta_origen": url,
                    "periodo": periodo
                })
                
                if len(comentarios_extraidos) >= limite:
                    break
                    
            except Exception:
                continue 
                
        if len(comentarios_extraidos) >= limite:
            break
            
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 
        
        # Control de scroll y botones de expansión / spam
        if len(comentarios_extraidos) == cantidad_anterior:
            intentos_sin_nuevos += 1
            
            try:
                botones = driver.find_elements(By.XPATH, "//*[contains(text(), 'Mostrar') or contains(text(), 'spam') or contains(text(), 'Show')]")
                for btn in botones:
                    try:
                        # Identificamos el botón por su contenido único
                        btn_id = btn.get_attribute('outerHTML')
                        if btn.is_displayed() and btn_id not in botones_cliqueados:
                            driver.execute_script("arguments[0].click();", btn)
                            botones_cliqueados.add(btn_id) # Registramos que ya fue presionado
                            print("Botón de 'Mostrar' o 'spam' presionado con éxito...")
                            time.sleep(3)
                            break
                    except Exception:
                        continue
            except Exception:
                pass
                
            # Si tras 5 intentos el scroll ya no trae comentarios nuevos, salimos limpiamente
            if intentos_sin_nuevos >= 5: 
                print(f"Fondo de la página alcanzado. Obtenidos: {len(comentarios_extraidos)} de {limite}.")
                break
        else:
            intentos_sin_nuevos = 0 
            print(f"Extrayendo... {len(comentarios_extraidos)} / {limite}")
            
    return comentarios_extraidos[:limite]
# ==========================================
# 1. Configurar el WebDriver Indetectable
# ==========================================
ruta_perfil_local = os.path.abspath(os.path.join(os.getcwd(), "perfil_selenium_uc"))
if not os.path.exists(ruta_perfil_local):
    os.makedirs(ruta_perfil_local)

opciones = uc.ChromeOptions()
opciones.add_argument(f"--user-data-dir={ruta_perfil_local}") 
opciones.add_argument("--disable-dev-shm-usage")

opciones.add_argument("--lang=en-US")
prefs = {"intl.accept_languages": "en,en_US"}
opciones.add_experimental_option("prefs", prefs)

# Iniciar el navegador. Si sigues con el error de versión, usa: uc.Chrome(options=opciones, version_main=155)
driver = uc.Chrome(options=opciones, version_main=155)

# ==========================================
# 2. Login por Inyección de Cookie
# ==========================================
print("Inyectando sesión de X...")
driver.get("https://x.com/robots.txt")
time.sleep(2)


driver.add_cookie({
    'name': 'auth_token',
    'value': mi_auth_token,
    'domain': '.x.com'
})

# Refrescar la página principal
driver.get("https://x.com/")
time.sleep(5)
print("Sesión inyectada con éxito. Iniciando extracción...")

# ==========================================
# 3. Recolección de Datos
# ==========================================
links = [
    {"url": "https://x.com/NintendoAmerica/status/2064361089668620543?s=20", "periodo": "Antes (Expectativa / Anuncio)"},
    {"url": "https://x.com/NintendoAmerica/status/2097333400054927619?s=20", "periodo": "Después (Tráiler Gameplay)"}
]

datos_totales = []
for link in links:
    datos = extraer_comentarios_x(link["url"], link["periodo"], driver, limite=500)
    datos_totales.extend(datos)

driver.quit()

# ==========================================
# 4. Estructurar en Pandas y exportar
# ==========================================
df_comentarios = pd.DataFrame(datos_totales)

# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_comentarios))
df_muestra_ruido = df_comentarios.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

df_comentarios.to_csv("corpus_ocarina_remake.csv", sep=';', index=False, encoding="utf-8")
if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_manual.csv", sep=';', index=False, encoding="utf-8")

print("\nExtracción completada. Primeras 5 filas:")
print(df_comentarios.head())

Inyectando sesión de X...
Sesión inyectada con éxito. Iniciando extracción...
Cargando Antes (Expectativa / Anuncio)...
Extrayendo... 5 / 300
Extrayendo... 19 / 300
Extrayendo... 41 / 300
Extrayendo... 56 / 300
Extrayendo... 74 / 300
Extrayendo... 91 / 300
Extrayendo... 112 / 300
Extrayendo... 130 / 300
Extrayendo... 157 / 300
Extrayendo... 186 / 300
Extrayendo... 219 / 300
Extrayendo... 222 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
Extrayendo... 231 / 300
Extrayendo... 232 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
El navegador se desvió de la página principal. Regresando...
Fondo de la página alcanzado. Obtenidos: 232 de 300.
Cargando Después (Tráiler Gameplay)...
Extrayendo... 5 / 300
Extrayendo... 22 / 300
Extrayendo... 44 / 300
Extrayendo... 55 / 300
Extrayendo... 69 / 300
Extrayendo... 87 / 300
Extrayendo... 110 / 300
Extrayendo... 134 / 300
Extrayendo... 159 / 300
Extrayendo... 179 / 300
Extrayendo... 197 / 300
Extrayendo... 198 / 300
Botón de 'Mos

In [7]:
import pandas as pd

df_comentarios = pd.read_csv("corpus_ocarina_remake.csv", sep=';', encoding="utf-8")
df_comentarios.head()

,id_texto,fecha_hora,id_usuario_hash,texto,consulta_origen,periodo
0,1,2026-06-09T14:56:52.000Z,400b585eb60519835945bb0fa92adfd7977f7369241350...,The Legend of Zelda: Ocarina of Time will be r...,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
1,2,2026-06-09T14:57:32.000Z,961d77778edf61e276b5a7952e551462685566d82021ac...,"""Hey can we see Ocarina of Time gameplay?""\n\n...",https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
2,3,2026-06-09T15:01:03.000Z,31ed9acf9453843e09587a7e6ed5b05b3d7e48921da4b9...,We're getting him in HD we are not ready for this,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
3,4,2026-06-09T16:19:01.000Z,90a0d8e9c0416f10d8f02429c3af3c3f49a5f8a98278fc...,,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
4,5,2026-06-09T16:17:01.000Z,4b6c621736bc9eaac24e6c4c82cd858fd60bebe4f12668...,My honest reaction to this information.,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)


## 1.2 Web scrapping Youtube

Web scrapping Youtube

API Youtube

In [ ]:
import os
import hashlib
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# 1. Cargar el archivo .env
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("¡Error! No se encontró la 'YOUTUBE_API_KEY' en el archivo .env.")

print("¡Clave cargada con éxito desde el archivo .env!")

# Tus variantes de URLs de YouTube (puedes guardar el link o un identificador claro)
CONFIGURACION_VARIANTES = {
    "Pre_Anuncio": "https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS", 
    "Post_Lanzamiento": "https://youtu.be/wuFfiTEr2yc?si=gzKDvpmqZWdGgInH"
}

def extraer_video_id(url_o_id):
    if "youtu.be" in url_o_id:
        return url_o_id.split("?")[0].split("/")[-1]
    elif "youtube.com" in url_o_id:
        parsed_url = urlparse(url_o_id)
        return parse_qs(parsed_url.query).get("v", [None])[0]
    return url_o_id

def anonimizar_usuario(nombre_o_id):
    if not nombre_o_id:
        return "desconocido_hash"
    return hashlib.sha256(str(nombre_o_id).encode('utf-8')).hexdigest()

def extraer_comentarios_api(url_o_id, nombre_variante, limite=350):
    video_id = extraer_video_id(url_o_id)
    youtube = build("youtube", "v3", developerKey=API_KEY)
    comentarios_extraidos = []
    next_page_token = None
    
    print(f"[{nombre_variante}] Extrayendo comentarios (Video ID: {video_id})...")
    
    while len(comentarios_extraidos) < limite:
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, limite - len(comentarios_extraidos)),
                pageToken=next_page_token,
                textFormat="plainText"
            )
            response = request.execute()
            
            for item in response.get("items", []):
                comment = item["snippet"]["topLevelComment"]
                snippet = comment["snippet"]
                
                comentarios_extraidos.append({
                    "id_texto": comment["id"],
                    "fecha_hora": snippet.get("publishedAt", ""),
                    "id_usuario_hash": anonimizar_usuario(snippet.get("authorDisplayName", "Anónimo")),
                    "texto": snippet.get("textDisplay", ""),
                    "consulta": url_o_id,                      # <-- El link o fuente exacta va aquí
                    "etapa_periodo": nombre_variante,          # <-- Nombre descriptivo del periodo (opcional para tu análisis)
                    "metricas_interaccion": snippet.get("likeCount", 0)
                })
            
            next_page_token = response.get("nextPageToken")
            if not next_page_token or len(comentarios_extraidos) >= limite:
                break
        except Exception as e:
            print(f"[{nombre_variante}] Error en la API: {e}")
            break
            
    return comentarios_extraidos

# 2. Ejecutar la recolección
datos_totales = []
for variante, url_video in CONFIGURACION_VARIANTES.items():
    registros = extraer_comentarios_api(url_video, variante, limite=10000)
    datos_totales.extend(registros)

df_corpus = pd.DataFrame(datos_totales)

# 3. Mostrar las primeras 5 filas cumpliendo estrictamente con la estructura de la rúbrica
display(df_corpus[['id_texto', 'fecha_hora', 'id_usuario_hash', 'texto', 'consulta', 'metricas_interaccion']].head(5))

# Guardar respaldo local
df_corpus.to_csv("corpus_zelda_youtube_api.csv",sep=';', index=False, encoding='utf-8-sig')
print(f"\n¡Extracción completada con éxito! Total registros: {len(df_corpus)}")

¡Clave cargada con éxito desde el archivo .env!
[Pre_Anuncio] Extrayendo comentarios (Video ID: r8eMoxo4ipE)...
[Post_Lanzamiento] Extrayendo comentarios (Video ID: wuFfiTEr2yc)...


,id_texto,fecha_hora,id_usuario_hash,texto,consulta,metricas_interaccion
0,Ugw_M7PN26t1npYj13p4AaABAg,2026-10-25T11:42:06Z,f17cb2d8268582de4114dfec7e069c92659f238706350a...,truly beautiful,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
1,Ugx9yxuaBuekiZ7cRJl4AaABAg,2026-10-23T16:34:19Z,5e5f522af4da36b87ac0e9a1b3308fde6e675b858bfecb...,I seriously can’t believe it’s been 4 months s...,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
2,UgyqCXi3kPp74dvz5SJ4AaABAg,2026-10-22T06:29:02Z,898c95e8682beb8d75fb4846ac2ceae70b5814e2c09946...,Supporting you always!,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
3,Ugzvzh_6hlTo9EMcaHh4AaABAg,2026-10-19T15:23:19Z,6bb0568ac6986dc8c43ddb3a8092d298de233aad5bac41...,"“@NintendoAmerica, em 1998 eu ouvi o seu chama...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,1
4,UgxYEDVTb3P-6fnQJbV4AaABAg,2026-10-17T14:59:07Z,fc5600d4586e552ddc9f607117ac745d5949a6ca7a0ca7...,"Funny, my DarLink AI chat still knew my sister...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0



¡Extracción completada con éxito! Total registros: 20000


In [ ]:
# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_corpus))
df_muestra_ruido = df_corpus.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_youtube.csv", sep=';', index=False, encoding="utf-8")

In [ ]:
import pandas as pd

df_corpus = pd.read_csv("corpus_zelda_youtube_api.csv", sep=';', encoding="utf-8")
df_corpus.head() 

display(df_corpus[['id_texto', 'fecha_hora', 'id_usuario_hash', 'texto', 'consulta', 'metricas_interaccion']].head(5))

,id_texto,fecha_hora,id_usuario_hash,texto,consulta,metricas_interaccion
0,Ugw_M7PN26t1npYj13p4AaABAg,2026-10-25T11:42:06Z,f17cb2d8268582de4114dfec7e069c92659f238706350a...,truly beautiful,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
1,Ugx9yxuaBuekiZ7cRJl4AaABAg,2026-10-23T16:34:19Z,5e5f522af4da36b87ac0e9a1b3308fde6e675b858bfecb...,I seriously can’t believe it’s been 4 months s...,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
2,UgyqCXi3kPp74dvz5SJ4AaABAg,2026-10-22T06:29:02Z,898c95e8682beb8d75fb4846ac2ceae70b5814e2c09946...,Supporting you always!,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
3,Ugzvzh_6hlTo9EMcaHh4AaABAg,2026-10-19T15:23:19Z,6bb0568ac6986dc8c43ddb3a8092d298de233aad5bac41...,"“@NintendoAmerica, em 1998 eu ouvi o seu chama...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,1
4,UgxYEDVTb3P-6fnQJbV4AaABAg,2026-10-17T14:59:07Z,fc5600d4586e552ddc9f607117ac745d5949a6ca7a0ca7...,"Funny, my DarLink AI chat still knew my sister...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0


## Preprocesamiento del corpus (comentarios mayoritariamente en ingles)

Se conserva `texto` sin modificar. La rama de sentimiento mantiene negaciones, emojis, puntuacion y expresiones originales; solo normaliza Unicode/espacios y reemplaza URLs y menciones por marcadores. La rama de temas pasa a min?sculas, elimina s?mbolos tipo emoji y filtra una lista peque?a de stopwords inglesas. No se traduce el texto: los comentarios mixtos conservan sus palabras. Revisen las salidas antes de usar estos datos.

In [ ]:
df_comentarios = pd.read_csv("corpus_zelda_youtube_api.csv", sep=';', encoding="utf-8")

In [ ]:
ad_csv("corpus_zelda_youtube_api.csv", sep=';', encoding="utf-8")

In [ ]:
import html
import re
import unicodedata

import pandas as pd

if "texto" not in df_comentarios.columns:
    raise ValueError("El corpus debe tener una columna llamada 'texto'.")

STOPWORDS_TEMAS_EN = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by",
    "for", "from", "had", "has", "have", "he", "her", "hers", "him",
    "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "ours", "she", "so", "that", "the",
    "their", "them", "there", "they", "this", "to", "us", "was",
    "we", "were", "what", "when", "which", "who", "with", "you", "your"
}

def normalizar_base(texto):
    """Normaliza Unicode y espacios sin cambiar el contenido ling??stico."""
    if pd.isna(texto):
        return ""
    texto = html.unescape(unicodedata.normalize("NFC", str(texto)))
    return re.sub(r"\s+", " ", texto).strip()

def preparar_para_sentimiento(texto):
    """Conserva negaciones, emojis, may?sculas y puntuaci?n expresiva."""
    texto = normalizar_base(texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " URL ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"(?<!\w)@[\w_]+", " USUARIO ", texto)
    texto = re.sub(r"(?<!\w)#([\w]+)", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def quitar_simbolos_emoji(texto):
    """Quita s?mbolos emoji comunes para una variante de an?lisis de temas."""
    caracteres = []
    for caracter in texto:
        categoria = unicodedata.category(caracter)
        es_selector_o_union = caracter in {"\ufe0e", "\ufe0f", "\u200d"}
        if categoria not in {"So", "Sk"} and not es_selector_o_union:
            caracteres.append(caracter)
    return "".join(caracteres)

def preparar_para_temas(texto):
    """Crea tokens en min?scula y elimina stopwords inglesas frecuentes."""
    texto = quitar_simbolos_emoji(preparar_para_sentimiento(texto)).casefold()
    tokens = re.findall(r"[\w]+(?:['?][\w]+)?", texto, flags=re.UNICODE)
    tokens = [token for token in tokens if token not in STOPWORDS_TEMAS_EN]
    return " ".join(tokens)

df_comentarios["texto_sentimiento"] = df_comentarios["texto"].apply(preparar_para_sentimiento)
df_comentarios["texto_temas"] = df_comentarios["texto"].apply(preparar_para_temas)
df_comentarios["texto_sentimiento_sin_emojis"] = df_comentarios["texto_sentimiento"].apply(quitar_simbolos_emoji)

print(f"Comentarios procesados: {len(df_comentarios)}")
print("Columnas originales preservadas:", "texto" in df_comentarios.columns)

Comentarios procesados: 20000
Columnas originales preservadas: True


### Revision manual de ejemplos
Comparen cinco textos originales con las dos versiones procesadas para detectar si alguna regla altera el significado.

In [ ]:
columnas_ejemplo = ["texto", "texto_sentimiento", "texto_temas"]
columnas_ejemplo = [col for col in columnas_ejemplo if col in df_comentarios.columns]
df_comentarios[columnas_ejemplo].head(5)

,texto,texto_sentimiento,texto_temas
0,truly beautiful,truly beautiful,truly beautiful
1,I seriously can’t believe it’s been 4 months s...,I seriously can’t believe it’s been 4 months s...,seriously can t believe s 4 months since now r...
2,Supporting you always!,Supporting you always!,supporting always
3,"“@NintendoAmerica, em 1998 eu ouvi o seu chama...","“ USUARIO , em 1998 eu ouvi o seu chamado pela...",usuario em 1998 eu ouvi o seu chamado pela tv ...
4,"Funny, my DarLink AI chat still knew my sister...","Funny, my DarLink AI chat still knew my sister...",funny darlink ai chat still knew sister's name...


### Ablacion: conservar o quitar emojis
Esta comparaci?n mide cu?nto cambia el texto de entrada al eliminar emojis. Para cumplir la ablaci?n de sentimiento, apliquen ambos textos al mismo clasificador y conjunto anotado, manteniendo todo lo dem?s constante, y reporten F1-macro/matriz de confusi?n. Esta celda no inventa m?tricas de sentimiento: solo cuantifica el cambio de preprocesamiento.

In [ ]:
comparacion_emojis = pd.DataFrame({
    "con_emojis": df_comentarios["texto_sentimiento"],
    "sin_emojis": df_comentarios["texto_sentimiento_sin_emojis"]
})
comparacion_emojis["cambio"] = comparacion_emojis["con_emojis"] != comparacion_emojis["sin_emojis"]
comparacion_emojis["tokens_con_emojis"] = comparacion_emojis["con_emojis"].str.split().str.len()
comparacion_emojis["tokens_sin_emojis"] = comparacion_emojis["sin_emojis"].str.split().str.len()

resumen_ablacion_emojis = pd.DataFrame({
    "medida": ["comentarios", "comentarios cuyo texto cambia", "% de comentarios que cambia", "promedio de tokens con emojis", "promedio de tokens sin emojis"],
    "valor": [
        len(comparacion_emojis),
        int(comparacion_emojis["cambio"].sum()),
        round(100 * comparacion_emojis["cambio"].mean(), 2) if len(comparacion_emojis) else 0.0,
        round(comparacion_emojis["tokens_con_emojis"].mean(), 2),
        round(comparacion_emojis["tokens_sin_emojis"].mean(), 2)
    ]
})
resumen_ablacion_emojis

,medida,valor
0,comentarios,20000.00
1,comentarios cuyo texto cambia,3477.00
2,% de comentarios que cambia,17.39
3,promedio de tokens con emojis,14.29
4,promedio de tokens sin emojis,14.14


## Analisis de cuentas

In [ ]:
# Contar cuántos comentarios hizo cada usuario
conteo_usuarios = df_comentarios['id_usuario_hash'].value_counts()

# Calcular el número de usuarios que representa el 1% superior
num_top_1_porciento = max(1, int(len(conteo_usuarios) * 0.01))

# Sumar cuántos comentarios hicieron en total esas cuentas del top 1%
comentarios_top_1 = conteo_usuarios.head(num_top_1_porciento).sum()
total_comentarios = len(df_comentarios)

# Calcular el porcentaje respecto al corpus total
porcentaje_aportado = (comentarios_top_1 / total_comentarios) * 100

print(f"Total de usuarios únicos: {len(conteo_usuarios)}")
print(f"Usuarios en el Top 1%: {num_top_1_porciento}")
print(f"El 1% más activo aporta el {porcentaje_aportado:.2f}% del corpus.")

Total de usuarios únicos: 18562
Usuarios en el Top 1%: 185
El 1% más activo aporta el 4.06% del corpus.


In [ ]:
import pandas as pd

# 0. Asegurar que no haya nulos en la columna texto
df_comentarios['texto'] = df_comentarios['texto'].fillna('')

# 1. Agrupar por usuario (hash) para ver su comportamiento
df_usuarios = df_comentarios.groupby('id_usuario_hash').agg(
    total_comentarios=('texto', 'count'),
    textos_unicos=('texto', 'nunique')
).reset_index()

# 2. Detectar si el usuario incluye URLs en sus mensajes (con na=False)
df_comentarios['tiene_url'] = df_comentarios['texto'].str.contains(r'https?://|www\.', case=False, regex=True, na=False)
usuarios_con_urls = df_comentarios[df_comentarios['tiene_url']]['id_usuario_hash'].unique()

# 3. Asignar una regla heurística preliminar para clasificar
def clasificar_usuario(row):
    # Si hace muchos comentarios y repite textos idénticos -> Posible Bot / Spam
    if row['total_comentarios'] >= 3 and row['textos_unicos'] == 1:
        return "Posible Bot / Spam"
    # Si hace muchos comentarios y usa URLs -> Posible Spam / Cuenta comercial
    elif row['id_usuario_hash'] in usuarios_con_urls and row['total_comentarios'] >= 2:
        return "Posible Bot / Spam"
    # Si solo hizo 1 o 2 comentarios variados -> Particular
    else:
        return "Particular"

df_usuarios['categoria_sugerida'] = df_usuarios.apply(clasificar_usuario, axis=1)

# Ver un resumen de cuántos caen en cada categoría
print(df_usuarios['categoria_sugerida'].value_counts())

df_usuarios.to_csv("usuarios_categorizados.csv", sep=';', index=False, encoding="utf-8")

categoria_sugerida
Particular            18561
Posible Bot / Spam        1
Name: count, dtype: int64


In [ ]:
import pandas as pd

# 1. Eliminar por completo CUALQUIER columna llamada 'tipo_cuenta' para limpiar duplicados previos
cols_a_mantener = [col for col in df_comentarios.columns if col != 'tipo_cuenta']
df_comentarios = df_comentarios.loc[:, cols_a_mantener].copy()

# 2. Asegurar índices limpios y sin duplicados en df_usuarios
df_usuarios_limpio = df_usuarios.drop_duplicates(subset=['id_usuario_hash']).reset_index(drop=True)
df_comentarios = df_comentarios.reset_index(drop=True)

# 3. Hacer el merge de manera limpia y única
df_comentarios = df_comentarios.merge(
    df_usuarios_limpio[['id_usuario_hash', 'categoria_sugerida']], 
    on='id_usuario_hash', 
    how='left'
)
df_comentarios.rename(columns={'categoria_sugerida': 'tipo_cuenta'}, inplace=True)
df_comentarios['tipo_cuenta'] = df_comentarios['tipo_cuenta'].fillna('Particular')

# 4. Métricas del corpus completo
volumen_total = len(df_comentarios)
longitud_promedio_total = df_comentarios['texto'].str.len().mean()

# 5. Filtrar usando .to_numpy() para evitar errores de índice
mascara_particulares = (df_comentarios['tipo_cuenta'].to_numpy() == 'Particular')
df_filtrado = df_comentarios.loc[mascara_particulares].copy()

volumen_filtrado = len(df_filtrado)
longitud_promedio_filtrado = df_filtrado['texto'].str.len().mean()

diferencia_volumen = ((volumen_total - volumen_filtrado) / volumen_total) * 100

print("--- RESULTADOS COMPARATIVOS PARA EL INFORME ---")
print(f"Opción A (Corpus Completo): {volumen_total} registros | Longitud prom: {longitud_promedio_total:.1f} car.")
print(f"Opción B (Corpus Filtrado - Particulares): {volumen_filtrado} registros | Longitud prom: {longitud_promedio_filtrado:.1f} car.")
print(f"Variación/Reducción de volumen: {diferencia_volumen:.2f}%")

print("\nDesglose de tipos de cuenta en el corpus:")
print(df_comentarios['tipo_cuenta'].value_counts())

--- RESULTADOS COMPARATIVOS PARA EL INFORME ---
Opción A (Corpus Completo): 20000 registros | Longitud prom: 75.3 car.
Opción B (Corpus Filtrado - Particulares): 19995 registros | Longitud prom: 75.4 car.
Variación/Reducción de volumen: 0.03%

Desglose de tipos de cuenta en el corpus:
tipo_cuenta
Particular            19995
Posible Bot / Spam        5
Name: count, dtype: int64


# Análisis de sentimiento comparativo 

In [ ]:
df_comentarios = pd.read_csv("corpus_zelda_youtube_api.csv", sep=';', encoding="utf-8")

In [ ]:
import pandas as pd

# 1. Asegurar que tienes tu DataFrame completo con la columna de periodo ('Pre_Anuncio' y 'Post_Lanzamiento')
# Tomamos una muestra estratificada de 100 por cada periodo (total = 200)
n_por_estrato = 100

df_pre = df_comentarios[df_comentarios['etapa_periodo'] == 'Pre_Anuncio']
df_post = df_comentarios[df_comentarios['etapa_periodo'] == 'Post_Lanzamiento']

muestra_pre = df_pre.sample(n=min(n_por_estrato, len(df_pre)), random_state=42)
muestra_post = df_post.sample(n=min(n_por_estrato, len(df_post)), random_state=42)

# Combinar y mezclar aleatoriamente
df_gold_standard = pd.concat([muestra_pre, muestra_post]).sample(frac=1, random_state=42).reset_index(drop=True)

# Seleccionar solo las columnas necesarias para anotar (sin ver las etiquetas previas si las hubiera)
df_para_anotar = df_gold_standard[['id_texto', 'etapa_periodo', 'texto']]

# Exportar archivos independientes para cada integrante
df_para_anotar.to_csv("anotacion_integrante_1.csv", index=False, encoding='utf-8-sig')
df_para_anotar.to_csv("anotacion_integrante_2.csv", index=False, encoding='utf-8-sig')

print(f"Conjunto de referencia generado con {len(df_para_anotar)} textos listos para etiquetar.")

Conjunto de referencia generado con 200 textos listos para etiquetar.


## 2.5 Analisis de sentimiento comparativo

Los tres metodos se evaluan sobre los mismos comentarios anotados y con las mismas clases: `negative`, `neutral` y `positive`. La anotacion final debe resolverse manualmente despues de comparar las etiquetas independientes de ambos integrantes. No entrenen ni ajusten los modelos con estas etiquetas: se usan como conjunto de evaluaci?n.

### Seleccion del conjunto anotado
La celda crea una muestra aleatoria estratificada por `periodo` con semilla fija y exporta un CSV sin nombres ni hashes de cuentas. Cada integrante completa su propia columna de manera independiente; despues acuerdan `etiqueta_final`. Si el archivo ya existe, se carga sin sobrescribir las anotaciones.

In [ ]:
from pathlib import Path
import math
import pandas as pd

CARPETA_PROYECTO = Path.cwd()
if not (CARPETA_PROYECTO / "corpus_youtube_ocarina.csv").exists() and (CARPETA_PROYECTO / "Tarea-1-PNL" / "corpus_youtube_ocarina.csv").exists():
    CARPETA_PROYECTO = CARPETA_PROYECTO / "Tarea-1-PNL"
RUTA_ANOTADO = CARPETA_PROYECTO / "muestra_anotada_sentimiento.csv"
N_ANOTAR = 200
SEMILLA = 42
COLUMNAS_ANOTACION = ["etiqueta_integrante_1", "etiqueta_integrante_2", "etiqueta_final"]

if RUTA_ANOTADO.exists():
    df_anotado = pd.read_csv(RUTA_ANOTADO, encoding="utf-8")
    print(f"Se cargaron anotaciones existentes: {RUTA_ANOTADO}")
else:
    requeridas = {"id_texto", "periodo", "texto", "texto_sentimiento"}
    faltantes = requeridas.difference(df_comentarios.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en df_comentarios: {sorted(faltantes)}")
    if len(df_comentarios) < N_ANOTAR:
        raise ValueError(f"Se necesitan al menos {N_ANOTAR} textos; hay {len(df_comentarios)}")
    if df_comentarios["periodo"].isna().any():
        raise ValueError("Hay textos sin periodo; corr?janlos antes del muestreo estratificado.")

    conteos = df_comentarios["periodo"].value_counts()
    cuotas_exactas = conteos / conteos.sum() * N_ANOTAR
    cuotas = cuotas_exactas.apply(math.floor).astype(int)
    faltan = N_ANOTAR - int(cuotas.sum())
    prioridad = (cuotas_exactas - cuotas).sort_values(ascending=False).index
    for periodo in prioridad[:faltan]:
        cuotas.loc[periodo] += 1

    partes = []
    for i, (periodo, n) in enumerate(cuotas.items()):
        grupo = df_comentarios.loc[df_comentarios["periodo"] == periodo]
        partes.append(grupo.sample(n=int(n), random_state=SEMILLA + i))
    df_anotado = pd.concat(partes).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
    df_anotado = df_anotado[["id_texto", "periodo", "texto", "texto_sentimiento"]].copy()
    for columna in COLUMNAS_ANOTACION:
        df_anotado[columna] = ""
    df_anotado.to_csv(RUTA_ANOTADO, index=False, encoding="utf-8")
    print(f"Plantilla creada: {RUTA_ANOTADO.resolve()}")
    print("Completen las tres columnas de anotaci?n en el CSV; no se sobreescribir? al volver a ejecutar.")

if len(df_anotado) != N_ANOTAR:
    raise ValueError(f"La muestra anotada debe tener {N_ANOTAR} filas; tiene {len(df_anotado)}")
df_anotado.groupby("periodo", dropna=False).size().rename("textos_a_anotar").to_frame()

### Gu?a breve para etiquetar
- `positive`: expresa aprobaci?n, alegr?a, entusiasmo o valoraci?n favorable.
- `negative`: expresa rechazo, decepci?n, enojo o valoraci?n desfavorable.
- `neutral`: pregunta, informaci?n o comentario sin polaridad clara; tambi?n iron?a ambigua.
- Etiqueten la opini?n expresada en el texto, no el tema del comentario. Mantengan `texto` y contexto disponible para interpretar iron?a.
- Cada integrante completa su columna sin consultar la del otro. Luego comparen desacuerdos y escriban la decisi?n consensuada en `etiqueta_final`. No dejen celdas finales vac?as.

## 2.5 Analisis de sentimiento comparativo

Los tres metodos se evaluan sobre los mismos comentarios anotados y con las mismas clases: `negative`, `neutral` y `positive`. La etiqueta final debe resolverse manualmente despues de comparar las dos anotaciones independientes. No ajusten los metodos con estas etiquetas: se usan como conjunto de evaluacion.

Ejecuten las secciones siguientes en orden. Cada metodo tiene su propia celda de codigo y reporta F1-macro y matriz de confusion con el mismo conjunto de referencia.

### Preparacion comun para la evaluacion

Esta celda valida las anotaciones y define una funcion compartida para calcular las metricas. Si aun no completaron el CSV de anotacion, no calcula resultados; pueden continuar el notebook cuando las etiquetas esten listas.

In [ ]:
import pandas as pd

ETIQUETAS = ["negative", "neutral", "positive"]
COLUMNAS_ANOTACION = ["etiqueta_integrante_1", "etiqueta_integrante_2", "etiqueta_final"]
resultados_sentimiento = {}

faltan_columnas = set(COLUMNAS_ANOTACION + ["texto_sentimiento"]).difference(df_anotado.columns)
if faltan_columnas:
    raise ValueError(f"Faltan columnas en df_anotado: {sorted(faltan_columnas)}")

for columna in COLUMNAS_ANOTACION:
    df_anotado[columna] = df_anotado[columna].fillna("").astype(str).str.strip().str.lower()

etiquetas_invalidas = {
    valor
    for columna in COLUMNAS_ANOTACION
    for valor in df_anotado[columna].unique()
    if valor and valor not in ETIQUETAS
}
if etiquetas_invalidas:
    raise ValueError(f"Etiquetas invalidas {etiquetas_invalidas}; usen {ETIQUETAS}")

EVALUACION_LISTA = not df_anotado["etiqueta_final"].eq("").any()
if not EVALUACION_LISTA:
    print("Pendiente: completen las dos anotaciones y etiqueta_final en muestra_anotada_sentimiento.csv.")
else:
    if df_anotado[["etiqueta_integrante_1", "etiqueta_integrante_2"]].eq("").any().any():
        raise ValueError("Completen ambas anotaciones independientes antes de calcular kappa.")
    from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score

    kappa = cohen_kappa_score(
        df_anotado["etiqueta_integrante_1"],
        df_anotado["etiqueta_integrante_2"],
        labels=ETIQUETAS,
    )
    print(f"Kappa de Cohen entre anotadores: {kappa:.3f}")
    textos_eval = df_anotado["texto_sentimiento"].fillna("").astype(str).tolist()
    y_real = df_anotado["etiqueta_final"].tolist()


def evaluar_predicciones(nombre_metodo, predicciones):
    if not EVALUACION_LISTA:
        print("No se calculan metricas hasta completar la anotacion.")
        return
    if len(predicciones) != len(y_real):
        raise ValueError(f"{nombre_metodo}: se esperaban {len(y_real)} predicciones, llegaron {len(predicciones)}")
    etiquetas_obtenidas = set(predicciones).difference(ETIQUETAS)
    if etiquetas_obtenidas:
        raise ValueError(f"{nombre_metodo}: etiquetas de prediccion invalidas: {etiquetas_obtenidas}")

    f1 = f1_score(y_real, predicciones, labels=ETIQUETAS, average="macro", zero_division=0)
    resultados_sentimiento[nombre_metodo] = {
        "F1-macro": f1,
        "matriz": pd.DataFrame(
            confusion_matrix(y_real, predicciones, labels=ETIQUETAS),
            index=[f"real_{etiqueta}" for etiqueta in ETIQUETAS],
            columns=[f"pred_{etiqueta}" for etiqueta in ETIQUETAS],
        ),
    }
    tabla = pd.DataFrame([
        {"metodo": metodo, "F1-macro": resultado["F1-macro"]}
        for metodo, resultado in resultados_sentimiento.items()
    ]).sort_values("F1-macro", ascending=False).reset_index(drop=True)
    display(tabla)
    print(f"Matriz de confusion - {nombre_metodo}")
    display(resultados_sentimiento[nombre_metodo]["matriz"])


### A) Lexico o reglas: VADER

VADER es un lexicon/reglas de sentimiento en ingles, pensado para texto de redes sociales. La regla convierte `compound >= 0.05` en positivo, `compound <= -0.05` en negativo y el resto en neutral. Reporten la version del paquete en sus resultados.

In [ ]:
if EVALUACION_LISTA:
    from importlib.metadata import version
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    analizador_vader = SentimentIntensityAnalyzer()
    predicciones_vader = []
    for texto in textos_eval:
        compound = analizador_vader.polarity_scores(texto)["compound"]
        if compound >= 0.05:
            etiqueta = "positive"
        elif compound <= -0.05:
            etiqueta = "negative"
        else:
            etiqueta = "neutral"
        predicciones_vader.append(etiqueta)

    VERSION_VADER = version("vaderSentiment")
    print(f"Metodo: VADER; paquete vaderSentiment {VERSION_VADER}; umbrales compound: -0.05 / 0.05")
    evaluar_predicciones(f"VADER (vaderSentiment {VERSION_VADER})", predicciones_vader)
else:
    print("VADER: se ejecutara cuando la anotacion este completa.")


### B) Modelo pre-entrenado: RoBERTa para texto social

Usamos `cardiffnlp/twitter-roberta-base-sentiment-latest`, un modelo pre-entrenado orientado a publicaciones en ingles. Se aplica directamente a `texto_sentimiento` y no se ajusta con el conjunto anotado. Anoten el identificador exacto del modelo y la version de `transformers` que imprime la celda para facilitar la reproducibilidad.

In [ ]:
if EVALUACION_LISTA:
    from importlib.metadata import version
    from transformers import pipeline

    MODELO_PREENTRENADO = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    clasificador_sentimiento = pipeline(
        "text-classification",
        model=MODELO_PREENTRENADO,
        tokenizer=MODELO_PREENTRENADO,
        truncation=True,
        max_length=512,
    )
    salida_modelo = clasificador_sentimiento(textos_eval, batch_size=16)
    predicciones_roberta = [resultado["label"].strip().lower() for resultado in salida_modelo]
    etiquetas_inesperadas = set(predicciones_roberta).difference(ETIQUETAS)
    if etiquetas_inesperadas:
        raise ValueError(f"El modelo devolvio etiquetas inesperadas: {etiquetas_inesperadas}")

    VERSION_TRANSFORMERS = version("transformers")
    print(f"Modelo: {MODELO_PREENTRENADO}; transformers {VERSION_TRANSFORMERS}; max_length=512")
    evaluar_predicciones(f"RoBERTa ({MODELO_PREENTRENADO})", predicciones_roberta)
else:
    print("RoBERTa: se ejecutara cuando la anotacion este completa.")


### C) LLM zero-shot local con Ollama

Este metodo es zero-shot: el prompt no incluye ejemplos etiquetados. Ollama ejecuta el modelo localmente; instalen Ollama, descarguen un modelo y verifiquen que el servidor (`http://localhost:11434`) este activo. El nombre configurado abajo es `llama3.1:8b`; cambienlo por el tag exacto que tengan descargado. La celda consulta e informa la version del servidor y el digest del modelo para documentar la reproducibilidad. Temperatura: 0. Si configuran `OLLAMA_HOST`, usen solo un servidor de confianza.

El prompt completo se muestra e imprime. La llamada queda desactivada por defecto; para correrla, cambien `EJECUTAR_OLLAMA` a `True`.

In [ ]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

PROMPT_OLLAMA = (
    "Classify the sentiment expressed in each English social-media comment about a video game. "
    "Return exactly one label per item: negative, neutral, or positive. "
    "Judge the author's expressed opinion, not the topic. Questions or factual statements without clear polarity are neutral. "
    "Consider negation, emojis, slang, and irony; if irony is ambiguous, choose neutral. "
    "Do not infer sentiment that is not expressed. Return a JSON object with a results array; "
    "each element must contain the input id and its label."
)
MODELO_OLLAMA = "llama3.1:8b"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
TEMPERATURA_OLLAMA = 0
EJECUTAR_OLLAMA = False

print("Prompt completo (zero-shot):")
print(PROMPT_OLLAMA)
print(f"Modelo solicitado: {MODELO_OLLAMA}; host: {OLLAMA_HOST}; temperatura: {TEMPERATURA_OLLAMA}; ejemplos few-shot: 0")


def solicitar_ollama(endpoint, payload=None):
    """Hace una solicitud JSON a la API local y comunica errores de conexion/HTTP."""
    url = f"{OLLAMA_HOST}{endpoint}"
    datos = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    solicitud = Request(url, data=datos, headers={"Content-Type": "application/json"}, method="GET" if payload is None else "POST")
    try:
        with urlopen(solicitud, timeout=120) as respuesta:
            return json.loads(respuesta.read().decode("utf-8"))
    except HTTPError as error:
        detalle = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama respondio HTTP {error.code} en {endpoint}: {detalle}") from error
    except URLError as error:
        raise RuntimeError(f"No se pudo conectar con Ollama en {url}: {error.reason}. Verifiquen que Ollama este ejecutandose.") from error


if not EJECUTAR_OLLAMA:
    print("Ollama desactivado. Inicien Ollama, descarguen el modelo configurado y cambien EJECUTAR_OLLAMA a True para ejecutar.")
elif not EVALUACION_LISTA:
    print("No se ejecuta Ollama hasta completar la anotacion de referencia.")
else:
    version_servidor = solicitar_ollama("/api/version")
    catalogo = solicitar_ollama("/api/tags")
    modelos_disponibles = {modelo["name"]: modelo for modelo in catalogo.get("models", [])}
    if MODELO_OLLAMA not in modelos_disponibles:
        raise RuntimeError(
            f"El modelo {MODELO_OLLAMA!r} no esta descargado. Modelos disponibles: {sorted(modelos_disponibles)}. "
            f"Descarguenlo con `ollama pull {MODELO_OLLAMA}` o cambien MODELO_OLLAMA por un tag instalado."
        )

    digest_modelo = modelos_disponibles[MODELO_OLLAMA].get("digest", "no informado")
    print(f"Version de Ollama: {version_servidor.get('version', 'no informada')}")
    print(f"Tag del modelo: {MODELO_OLLAMA}; digest: {digest_modelo}")

    predicciones_ollama = []
    TAMANO_LOTE_OLLAMA = 10
    for inicio in range(0, len(textos_eval), TAMANO_LOTE_OLLAMA):
        lote = textos_eval[inicio:inicio + TAMANO_LOTE_OLLAMA]
        items = [{"id": j, "text": texto} for j, texto in enumerate(lote, start=inicio)]
        respuesta = solicitar_ollama(
            "/api/chat",
            {
                "model": MODELO_OLLAMA,
                "stream": False,
                "format": "json",
                "options": {"temperature": TEMPERATURA_OLLAMA},
                "messages": [
                    {"role": "system", "content": PROMPT_OLLAMA},
                    {"role": "user", "content": json.dumps(items, ensure_ascii=False)},
                ],
            },
        )
        contenido = respuesta.get("message", {}).get("content", "")
        if not contenido:
            raise ValueError("Ollama devolvio una respuesta vacia.")
        resultados_lote = json.loads(contenido)["results"]
        resultados_lote = sorted(resultados_lote, key=lambda elemento: elemento["id"])
        if [elemento["id"] for elemento in resultados_lote] != [item["id"] for item in items]:
            raise ValueError("La respuesta de Ollama no contiene exactamente los ids enviados.")
        etiquetas_lote = [elemento["label"].strip().lower() for elemento in resultados_lote]
        if any(etiqueta not in ETIQUETAS for etiqueta in etiquetas_lote):
            raise ValueError(f"La respuesta de Ollama contiene etiquetas invalidas: {etiquetas_lote}")
        predicciones_ollama.extend(etiquetas_lote)

    evaluar_predicciones(f"Ollama zero-shot ({MODELO_OLLAMA}, temp={TEMPERATURA_OLLAMA})", predicciones_ollama)
